In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer

In [2]:
#loading the dataset
df = pd.read_csv('/content/20191226-reviews.csv')
df.head()

,asin,name,rating,date,verified,title,body,helpfulVotes
0,B0000SX2UC,Janet,3,"October 11, 2005",False,"Def not best, but not worst",I had the Samsung A600 for awhile which is abs...,1.0
1,B0000SX2UC,Luke Wyatt,1,"January 7, 2004",False,Text Messaging Doesn't Work,Due to a software issue between Nokia and Spri...,17.0
2,B0000SX2UC,Brooke,5,"December 30, 2003",False,Love This Phone,"This is a great, reliable phone. I also purcha...",5.0
3,B0000SX2UC,amy m. teague,3,"March 18, 2004",False,"Love the Phone, BUT...!","I love the phone and all, because I really did...",1.0
4,B0000SX2UC,tristazbimmer,4,"August 28, 2005",False,"Great phone service and options, lousy case!",The phone has been great for every purpose it ...,1.0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67986 entries, 0 to 67985
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   asin          67986 non-null  object 
 1   name          67983 non-null  object 
 2   rating        67986 non-null  int64  
 3   date          67986 non-null  object 
 4   verified      67986 non-null  bool   
 5   title         67957 non-null  object 
 6   body          67960 non-null  object 
 7   helpfulVotes  27215 non-null  float64
dtypes: bool(1), float64(1), int64(1), object(5)
memory usage: 3.7+ MB


In [4]:
df.shape

(67986, 8)

In [7]:
#checking for null values
df.isnull().sum()

,0
asin,0
name,3
rating,0
date,0
verified,0
title,29
body,26
helpfulVotes,40771


In [8]:
df.columns

Index(['asin', 'name', 'rating', 'date', 'verified', 'title', 'body',
       'helpfulVotes'],
      dtype='object')

In [10]:
# Select relevant columns
df = df[['body', 'rating']]

# Create sentiment labels
df['Sentiment'] = df['rating'].apply(lambda x: 1 if x > 3 else 0)

# Clean the text data
def clean_text(text):
    # Convert the input to string to handle potential floats
    text = str(text)
    text = re.sub(r"http\S+", "", text)  # Remove URLs
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove special characters
    text = text.lower()  # Convert to lowercase
    return text

df['Cleaned_Review'] = df['body'].apply(clean_text)

# Drop missing values, if any
df.dropna(subset=['Cleaned_Review'], inplace=True)

# Display the processed dataset
print(df.head())

                                                body  rating  Sentiment  \
0  I had the Samsung A600 for awhile which is abs...       3          0   
1  Due to a software issue between Nokia and Spri...       1          0   
2  This is a great, reliable phone. I also purcha...       5          1   
3  I love the phone and all, because I really did...       3          0   
4  The phone has been great for every purpose it ...       4          1   

                                      Cleaned_Review  
0  i had the samsung a for awhile which is absolu...  
1  due to a software issue between nokia and spri...  
2  this is a great reliable phone i also purchase...  
3  i love the phone and all because i really did ...  
4  the phone has been great for every purpose it ...  


In [12]:
# Define features and labels
X = df['Cleaned_Review']
y = df['Sentiment']

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training samples: {len(X_train)}, Test samples: {len(X_test)}")


Training samples: 54388, Test samples: 13598


In [13]:
# Convert text into numerical features using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)  # Use the top 5000 features

# Fit and transform the training data; transform the test data
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [14]:
# Train a Naive Bayes classifier
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Make predictions
y_pred = model.predict(X_test_vec)


In [15]:
# Evaluate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")

# Print classification report
print("Classification Report:\n", classification_report(y_test, y_pred))


Accuracy: 0.88
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.71      0.78      4276
           1       0.88      0.95      0.91      9322

    accuracy                           0.88     13598
   macro avg       0.87      0.83      0.85     13598
weighted avg       0.88      0.88      0.87     13598



In [17]:
# Pick some sample reviews from the dataset
sample_reviews = df['Cleaned_Review'].sample(5, random_state=42)

# Transform the samples using the vectorizer
sample_vec = vectorizer.transform(sample_reviews)

# Predict the sentiment
sample_predictions = model.predict(sample_vec)

# Display the reviews with their predictions
for review, prediction in zip(sample_reviews, sample_predictions):
    sentiment = "Positive" if prediction == 1 else "Negative"
    print(f"Review: {review}\nPredicted Sentiment: {sentiment}\n")


Review: nothing but flying stars for me here i had a ss had for years now its dosent work any more so i got an sedge and i always wanted one good to see that their just ass good as always i love you samsung always will loyal to the end
Predicted Sentiment: Positive

Review: really awesome phone i love it
Predicted Sentiment: Positive

Review: awesome phone  although its a lite version the phone is amazing super nice pictures and easy to use better than an iphone and samsung and cheaper price
Predicted Sentiment: Positive

Review: great camera window is amazing only compatible with verizon network will not send sms on other providers network battery is not accessible for replacement
Predicted Sentiment: Negative

Review: un gran telfono y el color es genial
Predicted Sentiment: Positive



In [19]:
# Custom input review
custom_reviews = [
    "The phone is amazing, I love it!",
    "This is the worst product I've ever bought.",
    "It works fine, but nothing extraordinary.",
]

# Clean the custom reviews (using the same cleaning function as earlier)
custom_reviews_cleaned = [clean_text(review) for review in custom_reviews]

# Transform the custom reviews using the vectorizer
custom_vec = vectorizer.transform(custom_reviews_cleaned)

# Predict the sentiment
custom_predictions = model.predict(custom_vec)

# Display the results
for review, prediction in zip(custom_reviews, custom_predictions):
    sentiment = "Positive" if prediction == 1 else "Negative"
    print(f"Review: {review}\nPredicted Sentiment: {sentiment}\n")


Review: The phone is amazing, I love it!
Predicted Sentiment: Positive

Review: This is the worst product I've ever bought.
Predicted Sentiment: Negative

Review: It works fine, but nothing extraordinary.
Predicted Sentiment: Positive



# **Dataset 2**

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report


In [22]:
#loading the dataset
df = pd.read_csv('/content/20191226-items.csv')
df.head()

,asin,brand,title,url,image,rating,reviewUrl,totalReviews,price,originalPrice
0,B0000SX2UC,NaN,Dual-Band / Tri-Mode Sprint PCS Phone w/ Voice...,https://www.amazon.com/Dual-Band-Tri-Mode-Acti...,https://m.media-amazon.com/images/I/2143EBQ210...,3.0,https://www.amazon.com/product-reviews/B0000SX2UC,14,0.00,0.0
1,B0009N5L7K,Motorola,Motorola I265 phone,https://www.amazon.com/Motorola-i265-I265-phon...,https://m.media-amazon.com/images/I/419WBAVDAR...,3.0,https://www.amazon.com/product-reviews/B0009N5L7K,7,49.95,0.0
2,B000SKTZ0S,Motorola,MOTOROLA C168i AT&T CINGULAR PREPAID GOPHONE C...,https://www.amazon.com/MOTOROLA-C168i-CINGULAR...,https://m.media-amazon.com/images/I/71b+q3ydkI...,2.7,https://www.amazon.com/product-reviews/B000SKTZ0S,22,99.99,0.0
3,B001AO4OUC,Motorola,Motorola i335 Cell Phone Boost Mobile,https://www.amazon.com/Motorola-i335-Phone-Boo...,https://m.media-amazon.com/images/I/710UO8gdT+...,3.3,https://www.amazon.com/product-reviews/B001AO4OUC,21,0.00,0.0
4,B001DCJAJG,Motorola,Motorola V365 no contract cellular phone AT&T,https://www.amazon.com/Motorola-V365-contract-...,https://m.media-amazon.com/images/I/61LYNCVrrK...,3.1,https://www.amazon.com/product-reviews/B001DCJAJG,12,149.99,0.0


In [24]:
# Focus on relevant columns
df = df[['rating', 'title']]

In [26]:
# Drop missing values
df.dropna(subset=['rating', 'title'], inplace=True)

# Map ratings to sentiment (e.g., Positive = 1, Negative = 0)
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)


In [27]:
# Text cleaning function
def clean_text(text):
    import re
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = text.lower()
    return text

# Apply text cleaning
df['Cleaned_Review'] = df['title'].apply(clean_text)

In [29]:
# Split data into training and testing sets
X = df['Cleaned_Review']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert text to numerical features using TF-IDF
vectorizer = TfidfVectorizer(max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)


In [30]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train_vec, y_train)
lr_predictions = lr_model.predict(X_test_vec)

# Evaluate
print("Logistic Regression Accuracy:", accuracy_score(y_test, lr_predictions))
print("Classification Report:\n", classification_report(y_test, lr_predictions))


Logistic Regression Accuracy: 0.75
Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.92      0.83        93
           1       0.76      0.43      0.55        51

    accuracy                           0.75       144
   macro avg       0.75      0.68      0.69       144
weighted avg       0.75      0.75      0.73       144



In [32]:
# Random Forest
rf_model = RandomForestClassifier()
rf_model.fit(X_train_vec, y_train)
rf_predictions = rf_model.predict(X_test_vec)

# Evaluate
print("Random Forest Accuracy:", accuracy_score(y_test, rf_predictions))
print("Classification Report:\n", classification_report(y_test, rf_predictions))


Random Forest Accuracy: 0.7638888888888888
Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.88      0.83        93
           1       0.72      0.55      0.62        51

    accuracy                           0.76       144
   macro avg       0.75      0.72      0.73       144
weighted avg       0.76      0.76      0.76       144



In [33]:
# SVM
svm_model = SVC()
svm_model.fit(X_train_vec, y_train)
svm_predictions = svm_model.predict(X_test_vec)

# Evaluate
print("SVM Accuracy:", accuracy_score(y_test, svm_predictions))
print("Classification Report:\n", classification_report(y_test, svm_predictions))


SVM Accuracy: 0.7569444444444444
Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.88      0.82        93
           1       0.71      0.53      0.61        51

    accuracy                           0.76       144
   macro avg       0.74      0.71      0.72       144
weighted avg       0.75      0.76      0.75       144



In [34]:
# Test samples
sample_reviews = [
    "The phone is amazing, works perfectly!",
    "Terrible product, not worth the money.",
    "Decent quality, but could be better.",
]

# Clean and transform the sample data
sample_cleaned = [clean_text(review) for review in sample_reviews]
sample_vec = vectorizer.transform(sample_cleaned)

# Predict sentiment with the trained Logistic Regression model (or any model)
sample_predictions = lr_model.predict(sample_vec)

# Output predictions
for review, sentiment in zip(sample_reviews, sample_predictions):
    sentiment_label = "Positive" if sentiment == 1 else "Negative"
    print(f"Review: {review}\nPredicted Sentiment: {sentiment_label}\n")


Review: The phone is amazing, works perfectly!
Predicted Sentiment: Negative

Review: Terrible product, not worth the money.
Predicted Sentiment: Negative

Review: Decent quality, but could be better.
Predicted Sentiment: Negative



In [36]:
print(df['sentiment'].value_counts())


sentiment
0    437
1    283
Name: count, dtype: int64


In [37]:
print(sample_cleaned)


['the phone is amazing works perfectly', 'terrible product not worth the money', 'decent quality but could be better']


In [38]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, lr_predictions))
print(classification_report(y_test, lr_predictions))


[[86  7]
 [29 22]]
              precision    recall  f1-score   support

           0       0.75      0.92      0.83        93
           1       0.76      0.43      0.55        51

    accuracy                           0.75       144
   macro avg       0.75      0.68      0.69       144
weighted avg       0.75      0.75      0.73       144



In [40]:
import warnings
from transformers import pipeline

sentiment_analyzer = pipeline('sentiment-analysis')

# Test samples
for review in sample_reviews:
    result = sentiment_analyzer(review)[0]
    print(f"Review: {review}")
    print(f"Predicted Sentiment: {result['label']} with score {result['score']:.4f}\n")


No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
Device set to use cpu


Review: The phone is amazing, works perfectly!
Predicted Sentiment: POSITIVE with score 0.9999

Review: Terrible product, not worth the money.
Predicted Sentiment: NEGATIVE with score 0.9998

Review: Decent quality, but could be better.
Predicted Sentiment: POSITIVE with score 0.9680

